# Bibliotecas

In [2]:
! pip install seaborn

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 KB 3.0 MB/s eta 0:00:00a 0:00:01


In [9]:
import os
from brevitas.export import export_qonnx
import onnx
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import gc
from brevitas.nn import QuantConv2d, QuantLinear, QuantReLU, TruncAvgPool2d
from brevitas.quant import Int32Bias
from configparser import ConfigParser
from torch.nn import Sequential
from brevitas.core.restrict_val import RestrictValueType
from brevitas.quant import Int8ActPerTensorFloat
from brevitas.quant import Int8WeightPerTensorFloat
from brevitas.quant import Uint8ActPerTensorFloat
from tqdm import tqdm
import copy
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from brevitas.export import export_qonnx
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.datatype import DataType
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from finn.util.visualization import showSrc
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames,RemoveStaticGraphInputs
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants
import finn.transformation.streamline.absorb as absorb
import finn.transformation.streamline.reorder as reorder
from finn.transformation.streamline import Streamline
from qonnx.transformation.double_to_single_float import DoubleToSingleFloat
from qonnx.transformation.change_datalayout import ChangeDataLayoutQuantAvgPool2d
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from finn.transformation.streamline.collapse_repeated import CollapseRepeatedMul
from qonnx.transformation.remove import RemoveIdentityOps
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.create_dataflow_partition import CreateDataflowPartition
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
import shutil
from finn.transformation.streamline.reorder import MoveScalarLinearPastInvariants, MakeMaxPoolNHWC
from qonnx.transformation.insert_topk import InsertTopK
from finn.transformation.streamline.absorb import AbsorbScalarMulAddIntoTopK,AbsorbAddIntoMultiThreshold,AbsorbMulIntoMultiThreshold
from torchvision.models import vgg11, VGG11_Weights
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score, confusion_matrix
from tqdm import tqdm
from brevitas import config
import warnings

# Dataset

In [4]:
BASE_PATH = 'DDR-dataset/DR_grading'
TRAIN_PATH = os.path.join(BASE_PATH, 'train')
VALID_PATH = os.path.join(BASE_PATH, 'valid')
TEST_PATH = os.path.join(BASE_PATH, 'test')

In [5]:
def load_labels(file_path):
    labels = pd.read_csv(file_path, sep=' ', header=None, names=['filename', 'label'])
    return labels

train_labels = load_labels(os.path.join(BASE_PATH, 'train.txt'))
valid_labels = load_labels(os.path.join(BASE_PATH, 'valid.txt'))
test_labels = load_labels(os.path.join(BASE_PATH, 'test.txt'))

In [6]:
class CustomDataset(Dataset):
    def __init__(self, labels, dir_path, transform=None):
        self.labels = labels
        self.dir_path = dir_path
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_name = os.path.join(self.dir_path, self.labels.iloc[idx, 0])
        image = Image.open(img_name).convert('RGB')
        label = int(self.labels.iloc[idx, 1])
        if self.transform:
            image = self.transform(image)
        return image, label

In [7]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(20),
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [8]:
BATCH_SIZE = 16  # Tamanho do batch

train_dataset = CustomDataset(train_labels, TRAIN_PATH, transform=train_transform)
valid_dataset = CustomDataset(valid_labels, VALID_PATH, transform=test_transform)
test_dataset = CustomDataset(test_labels, TEST_PATH, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Definindo modelo não quantizado

In [10]:
weights = VGG11_Weights.DEFAULT
model = vgg11(weights=weights)

num_ftrs = model.classifier[0].in_features  # 512*7*7
model.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 6)
)

Downloading: "https://download.pytorch.org/models/vgg11-8a719046.pth" to /tmp/home_dir/.cache/torch/hub/checkpoints/vgg11-8a719046.pth


  0%|          | 0.00/507M [00:00<?, ?B/s]

In [11]:
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 

# Definindo o modelo quantizado

In [ ]:
# Copyright (C) 2023, Advanced Micro Devices, Inc. All rights reserved.
# SPDX-License-Identifier: BSD-3-Clause

class CommonIntWeightPerTensorQuant(Int8WeightPerTensorFloat):
    scaling_min_val = 2e-16
    bit_width = None

class CommonIntWeightPerChannelQuant(CommonIntWeightPerTensorQuant):
    scaling_per_output_channel = True

class CommonIntActQuant(Int8ActPerTensorFloat):
    scaling_min_val = 2e-16
    bit_width = None
    restrict_scaling_type = RestrictValueType.LOG_FP

class CommonUintActQuant(Uint8ActPerTensorFloat):
    scaling_min_val = 2e-16
    bit_width = None
    restrict_scaling_type = RestrictValueType.LOG_FP

In [14]:
# BSD 3-Clause License
# Copyright (c) Alessandro Pappalardo 2019,
# Copyright (c) Soumith Chintala 2016,
# All rights reserved.

class QuantVGG11(nn.Module):
    def __init__(self, weight_bit_width, act_bit_width, num_classes=6):
        super(QuantVGG16, self).__init__()
        
        self.features = nn.Sequential(
            # Bloco 1
            QuantConv2d(3, 64, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Bloco 2
            QuantConv2d(64, 128, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Bloco 3
            QuantConv2d(128, 256, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(256, 256, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Bloco 4
            QuantConv2d(256, 512, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(512, 512, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Bloco 5
            QuantConv2d(512, 512, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(512, 512, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
        )


        self.classifier = nn.Sequential(
            QuantLinear(512*7*7, 
                        256, 
                        bias=True, 
                        weight_quant=CommonIntWeightPerChannelQuant,
                        weight_bit_width=weight_bit_width),
            QuantReLU(act_quant=CommonUintActQuant, 
                      bit_width=act_bit_width, 
                      return_quant_tensor=True),
            nn.Dropout(0.4),
            QuantLinear(256, 
                        num_classes, 
                        bias=True, 
                        weight_quant=CommonIntWeightPerChannelQuant,
                        weight_bit_width=weight_bit_width),
        )
        self._initialize_weights()

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

# Treinando o modelo não quantizado

In [19]:
def train_model(model, criterion, optimizer, train_loader, valid_loader, model_name, num_epochs=20):
    best_model_wts = model.state_dict()
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                data_loader = train_loader
            else:
                model.eval()
                data_loader = valid_loader
                
            running_loss = 0.0
            running_corrects = 0
            all_preds = []
            all_labels = []
            
            for inputs, labels in data_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
            
            epoch_loss = running_loss / len(data_loader.dataset)
            epoch_acc = running_corrects.double() / len(data_loader.dataset)
            
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            #torch.save(model.state_dict(), f'{model_name}_epoch{epoch+1}.pth')
            
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()
                torch.save(best_model_wts, model_name)
    
    print(f'Best val Acc: {best_acc:4f}')
    model.load_state_dict(best_model_wts)
    return model

In [16]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

Congelando o backbone para as primeiras épocas

In [17]:
for param in model.features.parameters():
    param.requires_grad = False

In [18]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(),lr=0.001)

In [ ]:
model = train_model(model, criterion, optimizer, train_loader, valid_loader,"best_vgg11.pth", num_epochs=5)

Epoch 1/5
----------
train Loss: 1.1532 Acc: 0.5625
val Loss: 0.8788 Acc: 0.6637
Epoch 2/5
----------
